# IPM geometry optimization

이 예는 PyAEDT를 사용하여 높은 토크를 달성하기 위한 최적의 기계 2D 형상을 찾는 방법을 보여줍니다.
을 찾아 높은 토크와 낮은 손실을 달성하는 방법을 보여줍니다.
이 예에서는 고정자 전류 각도와 자석의 다양한 재료에 대한 단일 값에 대해 지오메트리를 스윕하도록
를 스윕하도록 최적화 분석을 설정하는 방법을 보여줍니다.
그런 다음 토크 및 손실 결과를 .csv 파일로 내보냅니다.

Keywords: **Maxwell 2D**, **transient**, **motor**, **optimization**.

## Perform imports and define constants

Perform required imports.

In [1]:
import csv
import os
import tempfile
import time

import ansys.aedt.core
from ansys.aedt.core.examples.downloads import download_file


c:\Users\HAN_NDESKTOP\.ansys_python_venvs\pyAEDT_conda\Lib\site-packages\ansys\aedt\core\modeler\schematic.py:40: UserWarning: EMIT API is only available for Python 3.8-3.12.
  warnings.warn("EMIT API is only available for Python 3.8-3.12.")


Define constants.

In [2]:
AEDT_VERSION = "2025.2"
NUM_CORES = 4
NG_MODE = False  # Open AEDT UI when it is launched.

## Create temporary directory and download files

다운로드한 데이터 또는 덤프한 데이터를 저장할 수 있는 임시 디렉터리를
덤프한 데이터를 저장할 수 있는 임시 디렉터리를 만듭니다.
나중에 사용하기 위해 프로젝트 데이터를 검색하려는 경우,
임시 폴더 이름은 ``temp_folder.name``으로 지정합니다.

In [3]:
temp_folder = tempfile.TemporaryDirectory(suffix=".ansys")

## Download AEDT file example

Set the local temporary folder to export the AEDT file to.

In [4]:
aedt_file = download_file(
    source="maxwell_motor_optimization",
    name="IPM_optimization.aedt",
    local_path=temp_folder.name,
)

## Launch Maxwell 2D

Launch AEDT and Maxwell 2D after first setting up the project, the version and the graphical mode.

In [46]:
m2d.release_desktop()

PyAEDT INFO: Desktop has been released and closed.


True

In [5]:
m2d = ansys.aedt.core.Maxwell2d(
    project=aedt_file,
    version=AEDT_VERSION,
    new_desktop=True,
    non_graphical=NG_MODE,
)

PyAEDT INFO: Parsing C:\Users\HAN_NDESKTOP\AppData\Local\Temp\tmpg75b1zm4.ansys\maxwell_motor_optimization\IPM_optimization.aedt.
PyAEDT INFO: Python version 3.13.5 | packaged by conda-forge | (main, Jun 16 2025, 08:20:19) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\HAN_ND~1\AppData\Local\Temp\pyaedt_HAN_NDESKTOP_9ba62a39-f48f-4b3a-913f-e976aa1bff59.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: File C:\Users\HAN_NDESKTOP\AppData\Local\Temp\tmpg75b1zm4.ansys\maxwell_motor_optimization\IPM_optimization.aedt correctly loaded. Elapsed time: 0m 1sec
PyAEDT INFO: New AEDT session is starting on gRPC port 61034.
PyAEDT INFO: Electronics Desktop started on gRPC port: 61034 after 68.77553153038025 seconds.
PyAEDT I

## Design variables

파라메트릭 스윕에 사용할 머티리얼 배열을 정의합니다.

In [9]:
m2d["mat_sweep"] = '["XG196/96_2DSF1.000_X", "NdFe30", "NdFe35"]'
m2d["mat_index"] = 0

## Assign material array to magnets

디자인에서 기본적으로 ``XG196/96_2DSF1.000_X`` 머티리얼이 할당된 모든 자석을 가져옵니다.
위에 정의된 머티리얼 배열을 모든 자석에 할당합니다.

In [14]:
magnets = m2d.modeler.get_objects_by_material("XG196/96_2DSF1.000_X")

In [15]:
for mag in magnets:
    mag.material_name = "mat_sweep[mat_index]"

## Add parametric setup

Add a parametric setup made up of geometry variable sweep definitions and single value for the stator current angle.
Note: Step variations have been minimized to reduce the analysis time. If needed they can be increased by changing
the ``step`` argument.

In [16]:
param_sweep = m2d.parametrics.add(
    variable="bridge",
    start_point="0.5mm",
    variation_type="SingleValue",
)
param_sweep.add_variation(
    sweep_variable="din",
    start_point=78,
    end_point=80,
    step=10,
    units="mm",
    variation_type="LinearStep",
)
param_sweep.add_variation(
    sweep_variable="phase_advance",
    start_point=45,
    units="deg",
    variation_type="SingleValue",
)
param_sweep.add_variation(
    sweep_variable="Ipeak", start_point=200, units="A", variation_type="SingleValue"
)

True

Add material variation to the parametric setup and sweep the index of the material array defined above.

In [13]:
param_sweep.add_variation(
    sweep_variable="mat_index",
    start_point=0,
    end_point=2,
    step=1,
    variation_type="LinearStep",
)

True

## Alternative way to add a parametric setup from file

Suppose you have a .csv file with all the parameters to be swept defined in columns, such as:

# <img src="_static/param_sweep.png" alt="" width="400">

You can add a parametric setup from that file using the ``add_from_file`` method:

In [ ]:
# param_sweep_from_file = m2d.parametrics.add_from_file(csv_file_path)

## Analyze parametric sweep

To speed up the analysis, the time step is increased in the transient setup.
This can be done by modifying the ``TimeStep`` property of the transient setup.
Note: In a real case scenario, the time step should be: ``1/freq_e/360``.
To simulate a real case scenario, please comment out the following line.

In [17]:
m2d.setups[0].props["TimeStep"] = "1/freq_e/45"
param_sweep.analyze(cores=NUM_CORES)

PyAEDT INFO: Project IPM_optimization Saved correctly
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 2D correctly changed.
PyAEDT INFO: Solving Optimetrics
PyAEDT INFO: Design setup Parametric_ZD49W5 solved correctly in 0.0h 0.0m 37.0s
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 2D correctly changed.


True

## Post-processing

Create reports to get torque and loss results for all variations.
Create reports with all variations and with one variable at a time held constant.
This helps to visualize the influence of each variable on the torque and losses.
For the first torque report the ``din`` variable is held constant at 78mm.

In [18]:
report_torque_din_costant = m2d.post.create_report(
    expressions="Moving1.Torque",
    domain="Sweep",
    variations={
        "bridge": "All",
        "din": "78mm",
        "Ipeak": "All",
        "phase_advance": "All",
        "mat_index": "All",
    },
    primary_sweep_variable="Time",
    plot_type="Rectangular Plot",
    plot_name="torque_din_costant",
)

PyAEDT INFO: Parsing C:\Users\HAN_NDESKTOP\AppData\Local\Temp\tmpg75b1zm4.ansys\maxwell_motor_optimization\IPM_optimization.aedt.
PyAEDT INFO: File C:\Users\HAN_NDESKTOP\AppData\Local\Temp\tmpg75b1zm4.ansys\maxwell_motor_optimization\IPM_optimization.aedt correctly loaded. Elapsed time: 0m 0sec
PyAEDT INFO: aedt file load time 0.239027738571167
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec
PyAEDT WARNING: No report category provided. Automatically identified Transient


The second torque report has the ``mat_index`` variable held constant at 0.
In this case the material used for the magnets is ``"XG196/96_2DSF1.000_X"``.

In [19]:
report_torque_mat_costant = m2d.post.create_report(
    expressions="Moving1.Torque",
    domain="Sweep",
    variations={
        "bridge": "All",
        "din": "All",
        "Ipeak": "All",
        "phase_advance": "All",
        "mat_index": "0",
    },
    primary_sweep_variable="Time",
    plot_type="Rectangular Plot",
    plot_name="torque_mat_costant",
)

PyAEDT WARNING: No report category provided. Automatically identified Transient


The same approach is used to create reports for solid and core losses.

In [20]:
report_solid_loss_din_costant = m2d.post.create_report(
    expressions="SolidLoss",
    domain="Sweep",
    variations={
        "bridge": "All",
        "din": "78mm",
        "Ipeak": "All",
        "phase_advance": "All",
        "mat_index": "All",
    },
    primary_sweep_variable="Time",
    plot_type="Rectangular Plot",
    plot_name="solid_loss_din_costant",
)

PyAEDT WARNING: No report category provided. Automatically identified Transient


In [ ]:
report_solid_loss_mat_costant = m2d.post.create_report(
    expressions="SolidLoss",
    domain="Sweep",
    variations={
        "bridge": "All",
        "din": "All",
        "Ipeak": "All",
        "phase_advance": "All",
        "mat_index": "0",
    },
    primary_sweep_variable="Time",
    plot_type="Rectangular Plot",
    plot_name="solid_loss_mat_costant",
)

In [21]:
report_core_loss_din_costant = m2d.post.create_report(
    expressions="CoreLoss",
    domain="Sweep",
    variations={
        "bridge": "All",
        "din": "78mm",
        "Ipeak": "All",
        "phase_advance": "All",
        "mat_index": "All",
    },
    primary_sweep_variable="Time",
    plot_type="Rectangular Plot",
    plot_name="core_loss_din_costant",
)

PyAEDT WARNING: No report category provided. Automatically identified Transient


In [ ]:
report_core_loss_mat_costant = m2d.post.create_report(
    expressions="CoreLoss",
    domain="Sweep",
    variations={
        "bridge": "All",
        "din": "All",
        "Ipeak": "All",
        "phase_advance": "All",
        "mat_index": "0",
    },
    primary_sweep_variable="Time",
    plot_type="Rectangular Plot",
    plot_name="core_loss_mat_costant",
)

Get torque and loss solution data for all variations.

In [22]:
torque_data = m2d.post.get_solution_data(
    expressions=["Moving1.Torque"],
    setup_sweep_name=m2d.nominal_sweep,
    domain="Sweep",
    variations={
        "bridge": "All",
        "din": "All",
        "Ipeak": "All",
        "phase_advance": "All",
        "mat_index": "All",
    },
    primary_sweep_variable="Time",
    report_category="Standard",
)

PyAEDT INFO: Solution Data Correctly Loaded.
Time to initialize solution data:0.010411977767944336
Time to initialize solution data:0.013632774353027344


In [23]:
solid_loss_data = m2d.post.get_solution_data(
    expressions=["SolidLoss"],
    setup_sweep_name=m2d.nominal_sweep,
    domain="Sweep",
    variations={
        "bridge": "All",
        "din": "All",
        "Ipeak": "All",
        "phase_advance": "All",
        "mat_index": "All",
    },
    primary_sweep_variable="Time",
    report_category="Standard",
)

PyAEDT INFO: Solution Data Correctly Loaded.
Time to initialize solution data:0.01016855239868164
Time to initialize solution data:0.014434814453125


In [24]:
core_loss_data = m2d.post.get_solution_data(
    expressions=["CoreLoss"],
    setup_sweep_name=m2d.nominal_sweep,
    domain="Sweep",
    variations={
        "bridge": "All",
        "din": "All",
        "Ipeak": "All",
        "phase_advance": "All",
        "mat_index": "All",
    },
    primary_sweep_variable="Time",
    report_category="Standard",
)

PyAEDT INFO: Solution Data Correctly Loaded.
Time to initialize solution data:0.010994911193847656
Time to initialize solution data:0.014070749282836914


Calculate torque and loss average values for each variation and write data in a .csv file.

In [25]:
csv_data = []
for var in core_loss_data.variations:
    torque_data.active_variation = var
    core_loss_data.active_variation = var
    solid_loss_data.active_variation = var

    torque_values = torque_data.get_expression_data(formula="magnitude")[1]
    core_loss_values = core_loss_data.get_expression_data(formula="magnitude")[1]
    solid_loss_values = solid_loss_data.get_expression_data(formula="magnitude")[1]

    torque_data_average = sum(torque_values) / len(torque_values)
    core_loss_average = sum(core_loss_values) / len(core_loss_values)
    solid_loss_average = sum(solid_loss_values) / len(solid_loss_values)

    csv_data.append(
        {
            "active_variation": str(torque_data.active_variation),
            "average_torque": str(torque_data_average),
            "average_core_loss": str(core_loss_average),
            "average_solid_loss": str(solid_loss_average),
        }
    )

    with open(
        os.path.join(temp_folder.name, "motor_optimization.csv"), "w", newline=""
    ) as csvfile:
        fields = [
            "active_variation",
            "average_torque",
            "average_core_loss",
            "average_solid_loss",
        ]
        writer = csv.DictWriter(csvfile, fieldnames=fields)
        writer.writeheader()
        writer.writerows(csv_data)

In [ ]:
import ansys.aedt.core.visualization.post.common
m3CylinderPost=m3dCylinder.post

m3CylinderPost.plot_field('Mag_E','Region','Volume')


## 2D Plot

In [29]:
object_list=m2d.modeler.object_list

In [30]:
# 영역에 자속선 플롯 생성
# object_list는 섹션이 적용될 때 이전에 생성됩니다.
faces_reg = m2d.modeler.get_object_faces(object_list[1].name)  # Region


In [31]:
faces_reg

[2391]

In [33]:
plot1 = m2d.post.create_fieldplot_surface(
    assignment=faces_reg,
    quantity="Flux_Lines",
    plot_name="Flux_Lines",
)


PyAEDT INFO: Active Design set to M2D_Transient


In [45]:
m2d.post.plot_field(
    quantity="A_Vector", assignment=object_list, plot_cad_objs=False, mesh_on_fields=True,show=True
)

PyAEDT INFO: Active Design set to M2D_Transient


Widget(value='<iframe src="http://localhost:62463/index.html?ui=P_0x179d49f8e10_6&reconnect=auto" class="pyvis…

## Release AEDT

In [ ]:
m2d.save_project()
m2d.release_desktop()
# Wait 3 seconds to allow AEDT to shut down before cleaning the temporary directory.
time.sleep(3)

## Clean up

All project files are saved in the folder ``temp_folder.name``.
If you've run this example as a Jupyter notebook, you
can retrieve those project files. The following cell
removes all temporary files, including the project folder.

In [ ]:
temp_folder.cleanup()